<a href="https://colab.research.google.com/github/syoon7079/tjwn/blob/main/09_data_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1) 내 Google Drive 연결 — 원시자료는 각자 자기 Drive 에 둡니다
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/hdac_data' # 팀 공통 경로

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 2) 변수 정의표에 적은 변수만 읽기
!pip -q install pyreadstat
import pyreadstat
cols = ['age', 'sex'] # 변수 정의표의 원변수명으로 바꾸기
df, meta = pyreadstat.read_sav(f'/content/drive/MyDrive/hdac_data/HN24_ALL.sav',
usecols=cols)
print(df.shape) # 행 수와 열 수만 출력

(6997, 2)


In [3]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install pyreadstat
import pyreadstat
import pandas as pd
import numpy as np

FILE = '/content/drive/MyDrive/hdac_data/HN24_ALL.sav'
# 데이터는 읽지 않고 메타정보만 먼저 확인
_, meta0 = pyreadstat.read_sav(FILE, metadataonly=True)
print("전체 변수 수:", len(meta0.column_names))

# 대소문자 구분 없이 원변수명 찾기 (sav 파일마다 대/소문자가 다름)
lower_map = {c.lower(): c for c in meta0.column_names}

# 외식 빈도 변수 후보 찾기 (이용지침서 확인 후 확정)
print("\n[외식 관련 변수 후보]")
for c in meta0.column_names:
    label = meta0.column_names_to_labels.get(c, '')
    if 'out' in c.lower() or '외식' in str(label):
        print(c, '|', label)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
전체 변수 수: 813

[외식 관련 변수 후보]
L_OUT_FQ | 최근 1년 동안 평균 외식 빈도


#변수확인

In [4]:
EXPO_VAR = 'L_OUT_FQ'   # ← 이용지침서/셀1 결과로 확인 후 수정
BMI_VAR  = 'HE_BMI'     # 변수정의표에 'BMI'로 적혀 있으나 실제 원변수명 확인 필요

want = ['id', 'region', 'sex', 'age', 'edu', 'incm',
        'kstrata', 'psu', 'wt_itvex', 'wt_ntr',
        EXPO_VAR, BMI_VAR]

# 대소문자 자동 매칭
missing_names = [w for w in want if w.lower() not in lower_map]
print("파일에 없는 변수:", missing_names)

usecols = [lower_map[w.lower()] for w in want if w.lower() in lower_map]
df, meta = pyreadstat.read_sav(FILE, usecols=usecols)
df.columns = [c.lower() for c in df.columns]   # 이후 코드는 소문자로 통일
EXPO, BMI = EXPO_VAR.lower(), BMI_VAR.lower()

print(df.shape)
print(list(df.columns))

파일에 없는 변수: []
(6997, 12)
['id', 'region', 'psu', 'sex', 'age', 'incm', 'edu', 'wt_itvex', 'wt_ntr', 'kstrata', 'he_bmi', 'l_out_fq']


In [5]:
print("=== dtype / 결측 ===")
chk = pd.DataFrame({
    'dtype': df.dtypes,
    '결측수': df.isna().sum(),
    '결측률(%)': (df.isna().mean()*100).round(2),
    '고유값수': df.nunique()
})
print(chk)

print("\n=== 연속형 요약통계 ===")
print(df[[BMI, 'age', 'wt_ntr', 'wt_itvex']].describe().round(3))

print("\n=== 변수 라벨 ===")
for c in df.columns:
    orig = lower_map.get(c, c)
    print(c, '|', meta.column_names_to_labels.get(orig, ''))

print("\n=== 값 라벨(원코드 의미) ===")
for c in [EXPO, 'sex', 'edu', 'incm', 'region']:
    orig = lower_map.get(c, c)
    print(c, meta.variable_value_labels.get(orig, '값 라벨 없음'))

=== dtype / 결측 ===
            dtype  결측수  결측률(%)  고유값수
id         object    0    0.00  6997
region    float64    0    0.00    17
psu        object    0    0.00   192
sex       float64    0    0.00     2
age       float64    0    0.00    80
incm      float64   45    0.64     4
edu       float64  233    3.33     4
wt_itvex  float64    0    0.00  3469
wt_ntr    float64  192    2.74  3512
kstrata   float64    0    0.00    27
he_bmi    float64  116    1.66  6710
l_out_fq  float64  192    2.74     7

=== 연속형 요약통계 ===
         he_bmi       age     wt_ntr   wt_itvex
count  6881.000  6997.000   6805.000   6997.000
mean     23.432    48.101   7571.203   7363.446
std       4.315    22.072   4000.028   3807.130
min      11.564     1.000    319.604    314.854
25%      20.620    31.000   4768.548   4738.511
50%      23.271    52.000   6784.064   6665.953
75%      26.000    66.000   9318.555   9024.211
max      48.684    80.000  32117.498  30575.038

=== 변수 라벨 ===
id | 개인 아이디
region | 시도
psu | 조사구번호

#원코드별 빈도표

In [6]:
def freq_table(data, var, wcol='wt_ntr'):
    t = data[var].value_counts(dropna=False).sort_index()
    w = data.groupby(var, dropna=False)[wcol].sum()
    out = pd.DataFrame({
        '비가중 N': t,
        '비가중 %': (t / t.sum() * 100).round(2),
        '가중 합계': w.round(0),
        '가중 %': (w / w.sum() * 100).round(2),
    })
    return out

for v in [EXPO, 'sex', 'edu', 'incm', 'region']:
    print(f"\n===== {v} 원코드별 빈도 =====")
    print(freq_table(df, v))


===== l_out_fq 원코드별 빈도 =====
          비가중 N  비가중 %       가중 합계   가중 %
l_out_fq                                 
1.0         246   3.52   2271128.0   4.41
2.0        1083  15.48   9559372.0  18.55
3.0        1477  21.11  12270776.0  23.82
4.0         829  11.85   6710080.0  13.02
5.0        1457  20.82  10413694.0  20.21
6.0        1199  17.14   7431140.0  14.42
7.0         514   7.35   2865844.0   5.56
NaN         192   2.74         0.0   0.00

===== sex 원코드별 빈도 =====
     비가중 N  비가중 %       가중 합계  가중 %
sex                                
1.0   3070  43.88  25759551.0  50.0
2.0   3927  56.12  25762484.0  50.0

===== edu 원코드별 빈도 =====
     비가중 N  비가중 %       가중 합계   가중 %
edu                                 
1.0   1759  25.14  10559319.0  20.49
2.0    742  10.60   4875761.0   9.46
3.0   1930  27.58  15033474.0  29.18
4.0   2333  33.34  20050650.0  38.92
NaN    233   3.33   1002832.0   1.95

===== incm 원코드별 빈도 =====
      비가중 N  비가중 %       가중 합계   가중 %
incm                             

In [7]:
# 원코드 → 분석용 3범주 (0: 주 1~2회 이하, 1: 주 3~4회, 2: 주 5회 이상)
# ⚠️ 반드시 값 라벨 출력(셀 3)과 대조해서 수정하세요.
EXPO_MAP = {
    1: 2,   # 예: 주 5회 이상
    2: 1,   # 예: 주 3~4회
    3: 0,   # 예: 주 1~2회
    4: 0,   # 예: 월 1~3회
    5: 0,   # 예: 거의 안 함
}
NA_CODES = [8, 9, 99, 888, 999]   # 비해당/모름/무응답

d = df.copy()
for v in [EXPO, 'edu', 'incm']:
    d[v] = d[v].replace(NA_CODES, np.nan)

d['expo_grp'] = d[EXPO].map(EXPO_MAP)   # 매핑에 없는 코드는 자동 NaN

print(pd.crosstab(d[EXPO], d['expo_grp'], dropna=False))

expo_grp   0.0   1.0  2.0   NaN
l_out_fq                       
1.0          0     0  246     0
2.0          0  1083    0     0
3.0       1477     0    0     0
4.0        829     0    0     0
5.0       1457     0    0     0
6.0          0     0    0  1199
7.0          0     0    0   514
NaN          0     0    0   192


#분석 대상 선정에서 단계별 남는 인원

In [8]:
steps = []
def log(name, data):
    steps.append({'단계': name, '남는 인원(N)': len(data)})

# 1단계: 전체 참여자
s1 = d.copy(); log('1단계: 전체 참여자', s1)

# 2단계: 포함기준 (검진·영양 모두 참여 = 통합가중치 존재, 기본변수 연령·성별 보유)
s2 = s1[s1['wt_ntr'].notna() & (s1['wt_ntr'] > 0)
        & s1['age'].notna() & s1['sex'].notna()]
log('2단계: 포함기준(통합가중치>0, 연령·성별 존재)', s2)

# 3단계: 제외기준 - 주요변수 및 복합표본설계 변수 결측 제거
s3a = s2[s2['expo_grp'].notna()];                         log('3-1: 외식빈도 결측 제외', s3a)
s3b = s3a[s3a[BMI].notna()];                              log('3-2: BMI 결측 제외', s3b)
s3c = s3b[s3b[['kstrata', 'psu', 'wt_ntr']].notna().all(axis=1)]
log('3-3: 층·집락·가중치 결측 제외', s3c)

# 4단계: 공변량 결측 제거 → 최종 분석대상
covs = ['region', 'sex', 'age', 'edu', 'incm']
s4 = s3c[s3c[covs].notna().all(axis=1)]
log('4단계: 공변량(지역·성별·연령·교육·소득) 결측 제외 = 최종', s4)

flow = pd.DataFrame(steps)
flow['직전 대비 감소'] = flow['남는 인원(N)'].diff().fillna(0).astype(int)
print(flow.to_string(index=False))

final = s4.copy()

                                 단계  남는 인원(N)  직전 대비 감소
                        1단계: 전체 참여자      6997         0
       2단계: 포함기준(통합가중치>0, 연령·성별 존재)      6805      -192
                    3-1: 외식빈도 결측 제외      5092     -1713
                     3-2: BMI 결측 제외      5046       -46
                3-3: 층·집락·가중치 결측 제외      5046         0
4단계: 공변량(지역·성별·연령·교육·소득) 결측 제외 = 최종      4945      -101


#가중치·층·집락 확인

In [9]:
print("=== 최종 분석대상 설계변수 ===")
print("N:", len(final))
print("층(kstrata) 수:", final['kstrata'].nunique())
print("집락(psu) 수:", final['psu'].nunique())
print("가중치 요약:\n", final['wt_ntr'].describe().round(4))
print("가중치 합계(추정 모집단 규모):", round(final['wt_ntr'].sum()))

# 층별 PSU 수 확인 (분산추정 시 층당 PSU가 1개면 문제 발생)
psu_per_strata = final.groupby('kstrata')['psu'].nunique()
print("\n층당 PSU 수 분포:\n", psu_per_strata.describe())
lonely = psu_per_strata[psu_per_strata < 2]
print("PSU가 1개뿐인 층(lonely PSU):", len(lonely), "개")
print(lonely)

# 참고: 검진 가중치와의 비교
print("\nwt_itvex 요약:\n", final['wt_itvex'].describe().round(4))
print("wt_ntr ≤ wt_itvex 인 비율:",
      (final['wt_ntr'] <= final['wt_itvex']).mean().round(3))

# 노출군별 가중 평균 BMI (사전 확인용)
g = final.groupby('expo_grp')
res = pd.DataFrame({
    'N': g.size(),
    '가중 평균 BMI': g.apply(lambda x: np.average(x[BMI], weights=x['wt_ntr'])).round(2),
    '비가중 평균 BMI': g[BMI].mean().round(2),
})
print("\n=== 외식 빈도군별 BMI ===")
print(res)

=== 최종 분석대상 설계변수 ===
N: 4945
층(kstrata) 수: 27
집락(psu) 수: 192
가중치 요약:
 count     4945.0000
mean      8123.2071
std       4110.1807
min        319.6044
25%       5285.4784
50%       7393.9010
75%       9971.1142
max      32117.4983
Name: wt_ntr, dtype: float64
가중치 합계(추정 모집단 규모): 40169259

층당 PSU 수 분포:
 count    27.000000
mean      7.111111
std       5.963565
min       2.000000
25%       3.500000
50%       6.000000
75%       7.500000
max      25.000000
Name: psu, dtype: float64
PSU가 1개뿐인 층(lonely PSU): 0 개
Series([], Name: psu, dtype: int64)

wt_itvex 요약:
 count     4945.0000
mean      7911.7095
std       3912.6199
min        314.8535
25%       5206.9117
50%       7218.3837
75%       9686.8297
max      30575.0375
Name: wt_itvex, dtype: float64
wt_ntr ≤ wt_itvex 인 비율: 0.556

=== 외식 빈도군별 BMI ===
             N  가중 평균 BMI  비가중 평균 BMI
expo_grp                             
0.0       3653      23.49       23.25
1.0       1055      23.41       23.00
2.0        237      24.46       24.21


/tmp/ipykernel_8189/3177535787.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  '가중 평균 BMI': g.apply(lambda x: np.average(x[BMI], weights=x['wt_ntr'])).round(2),
